In [75]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_curve, roc_auc_score
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import xgboost as xgb
import joblib

In [48]:
from google.colab import drive
drive.mount("/content/drive")
DATASET_PATH = "/content/drive/MyDrive/Torgo"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [49]:
audio_files = []
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith('.wav'):
            audio_files.append(os.path.join(root, file))

print(f" Found {len(audio_files)} audio files")


 Found 15939 audio files


In [50]:
control_folders = ['F_Con', 'M_Con']    # Control ke folders
dysarthric_folders = ['F_Dys', 'M_Dys'] # Dysarthric ke folders

data = []

for file in audio_files:
    label = None
    speaker_group = None

    # Check which folder the file belongs to
    if 'F_Con' in file:
        label = 'Control'
        speaker_group = 'F_Con'
    elif 'M_Con' in file:
        label = 'Control'
        speaker_group = 'M_Con'
    elif 'F_Dys' in file:
        label = 'Dysarthric'
        speaker_group = 'F_Dys'
    elif 'M_Dys' in file:
        label = 'Dysarthric'
        speaker_group = 'M_Dys'

    if label:
        data.append([file, label, speaker_group])

df = pd.DataFrame(data, columns=['filepath', 'label', 'speaker_group'])

print(f"\n Dataset Info:")
print(f"Total: {len(df)} files")
print(f"Control: {len(df[df.label=='Control'])} files")
print(f"Dysarthric: {len(df[df.label=='Dysarthric'])} files")

print(f"\n Files per group:")
print(df['speaker_group'].value_counts())


 Dataset Info:
Total: 15939 files
Control: 9760 files
Dysarthric: 6179 files

 Files per group:
speaker_group
M_Con    6359
M_Dys    3788
F_Con    3401
F_Dys    2391
Name: count, dtype: int64


In [51]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
df['label'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0], colors=['green', 'red'])
axes[0].set_title('Class Distribution')

# Bar chart
df['label'].value_counts().plot(kind='bar', ax=axes[1], color=['green', 'red'])
axes[1].set_title('Number of Samples per Class')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()


In [52]:
durations = []
for file in tqdm(df['filepath'][:50], desc="Analyzing durations"):
    y, sr = librosa.load(file, sr=None)
    durations.append(len(y)/sr)

plt.figure(figsize=(10, 4))
plt.hist(durations, bins=20, color='blue', alpha=0.7)
plt.xlabel('Duration (seconds)')
plt.ylabel('Number of files')
plt.title('Audio Duration Distribution')
plt.axvline(x=3, color='red', linestyle='--', label='3 seconds')
plt.legend()
plt.show()

Analyzing durations: 100%|██████████| 50/50 [00:02<00:00, 20.19it/s]


In [53]:
def get_features(file_path, duration=3, sr=16000):
    """Extract simple features from audio"""
    try:
        # Load audio
        y, _ = librosa.load(file_path, sr=sr, duration=duration)

        # Pad if needed
        if len(y) < duration * sr:
            y = np.pad(y, (0, duration * sr - len(y)))
        else:
            y = y[:duration * sr]

        # Remove silence
        y, _ = librosa.effects.trim(y, top_db=20)

        # MFCC (13 coefficients)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        # Extract features: mean and std of each MFCC
        features = []
        for i in range(13):
            features.append(np.mean(mfcc[i]))
            features.append(np.std(mfcc[i]))

        # Add pitch
        pitches, _ = librosa.piptrack(y=y, sr=sr)
        pitch_vals = pitches[pitches > 0]
        if len(pitch_vals) > 0:
            features.append(np.mean(pitch_vals))
            features.append(np.std(pitch_vals))
        else:
            features.extend([0, 0])

        # Add energy
        energy = np.sum(y**2) / len(y)
        features.append(energy)

        return np.array(features)

    except:
        return np.zeros(29)  # 13*2 + 2 + 1 = 29 features

print(" Extracting features from all audio files...")

 Extracting features from all audio files...


In [54]:
X = []
y_labels = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    features = get_features(row['filepath'])
    X.append(features)
    y_labels.append(1 if row['label'] == 'Dysarthric' else 0)

X = np.array(X)
y = np.array(y_labels)

print(f"\n Features extracted: {X.shape}")
print(f"Features per audio: {X.shape[1]}")

100%|██████████| 15939/15939 [09:31<00:00, 27.91it/s]



 Features extracted: (15939, 29)
Features per audio: 29


In [55]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


Train: 11157 | Val: 2391 | Test: 2391


In [56]:
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print(f"After balancing: {len(X_train_bal)} samples")


After balancing: 13664 samples


In [57]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [58]:
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.AUC(name='auc')])


In [59]:
history = model.fit(
    X_train_scaled, y_train_bal,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[
        EarlyStopping(patience=10, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.5, patience=5)
    ]
)

Epoch 1/50
427/427 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7091 - auc: 0.7908 - loss: 0.5481 - val_accuracy: 0.7988 - val_auc: 0.8766 - val_loss: 0.4385 - learning_rate: 0.0010
Epoch 2/50
427/427 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7964 - auc: 0.8820 - loss: 0.4306 - val_accuracy: 0.8365 - val_auc: 0.9133 - val_loss: 0.3591 - learning_rate: 0.0010
Epoch 3/50
427/427 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8230 - auc: 0.9078 - loss: 0.3832 - val_accuracy: 0.8607 - val_auc: 0.9359 - val_loss: 0.3202 - learning_rate: 0.0010
Epoch 4/50
427/427 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8473 - auc: 0.9272 - loss: 0.3445 - val_accuracy: 0.8724 - val_auc: 0.9460 - val_loss: 0.2907 - learning_rate: 0.0010
Epoch 5/50
427/427 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8573 - auc: 0.9358 - loss: 0.3235 - val_accuracy: 0.8879 - val_auc: 0.9543 - val_loss: 0.2703 - learning_rate: 0.0010
Epoch 6/50
427/427 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8642 - auc: 0.

In [60]:
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")

if accuracy >= 0.90:
    print(" SUCCESS! Target achieved (90%+ accuracy)")
else:
    print(f" Accuracy {accuracy:.2%} - try adjusting features")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy: 0.9364
F1-Score: 0.9195
 SUCCESS! Target achieved (90%+ accuracy)


In [63]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Val Loss', marker='s')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [64]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Control', 'Dysarthric'],
            yticklabels=['Control', 'Dysarthric'])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_ylabel('True Label')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Control', 'Dysarthric'],
            yticklabels=['Control', 'Dysarthric'])
axes[1].set_title('Confusion Matrix (Percentage)')

plt.tight_layout()
plt.show()

In [65]:
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc_score = roc_auc_score(y_test, y_pred_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [66]:
control_sample = df[df.label=='Control']['filepath'].iloc[0]
dys_sample = df[df.label=='Dysarthric']['filepath'].iloc[0]

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

y1, sr1 = librosa.load(control_sample, sr=16000)
librosa.display.waveshow(y1, sr=sr1, ax=axes[0])
axes[0].set_title('Control Speech - Normal Waveform')
axes[0].set_ylabel('Amplitude')

y2, sr2 = librosa.load(dys_sample, sr=16000)
librosa.display.waveshow(y2, sr=sr2, ax=axes[1])
axes[1].set_title('Dysarthric Speech - Disrupted Waveform')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

In [67]:
joblib.dump(model, 'speech_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print(" Model saved as 'speech_model.pkl'")
print(" Scaler saved as 'scaler.pkl'")


 Model saved as 'speech_model.pkl'
 Scaler saved as 'scaler.pkl'


In [68]:
from google.colab import files

files.download('speech_model.pkl')
files.download('scaler.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [69]:
def predict_single_audio(audio_path, model, scaler, threshold=0.5):
    """
    Predict if a single audio file is Dysarthric or Control

    Parameters:
    - audio_path: path to audio file
    - model: trained Keras model
    - scaler: fitted StandardScaler
    - threshold: decision threshold (default 0.5)

    Returns:
    - prediction: 'Dysarthric' or 'Control'
    - confidence: confidence score (0-100)
    - probability: raw probability
    - features: extracted features
    """
    try:
        # Extract features
        features = get_features(audio_path)

        # Check if features are valid
        if np.sum(np.abs(features)) == 0:
            return None, 0, 0, None

        # Scale features
        features_scaled = scaler.transform(features.reshape(1, -1))

        # Make prediction
        probability = float(model.predict(features_scaled, verbose=0)[0][0])

        # Apply threshold
        prediction = 'Dysarthric' if probability > threshold else 'Control'

        # Calculate confidence (percentage)
        confidence = probability * 100 if probability > threshold else (1 - probability) * 100

        return prediction, confidence, probability * 100, features

    except Exception as e:
        print(f"Error in prediction: {e}")
        return None, 0, 0, None


In [70]:
def test_random_prediction():
    """Test prediction on a random audio file from dataset"""

    # Get random audio files
    control_files = df[df.label == 'Control']['filepath'].tolist()
    dysarthric_files = df[df.label == 'Dysarthric']['filepath'].tolist()
    if control_files:
        test_file = np.random.choice(control_files)
        pred, conf, prob, _ = predict_single_audio(test_file, model, scaler)
        print(f"\n Control Sample: {os.path.basename(test_file)}")
        print(f" Actual: Control")
        print(f" Predicted: {pred}")
        print(f" Confidence: {conf:.2f}%")
        print(f" Probability: {prob:.2f}%")

    if dysarthric_files:
          test_file = np.random.choice(dysarthric_files)
          pred, conf, prob, _ = predict_single_audio(test_file, model, scaler)
          print(f"\n Dysarthric Sample: {os.path.basename(test_file)}")
          print(f" Actual: Dysarthric")
          print(f" Predicted: {pred}")
          print(f" Confidence: {conf:.2f}%")
          print(f" Probability: {prob:.2f}%")

In [71]:
test_random_prediction()



 Control Sample: wav_headMic_FC02S03_0276.wav
 Actual: Control
 Predicted: Control
 Confidence: 96.94%
 Probability: 3.06%

 Dysarthric Sample: wav_arrayMic_M04S02_0097.wav
 Actual: Dysarthric
 Predicted: Dysarthric
 Confidence: 99.90%
 Probability: 99.90%


In [72]:
# ============================================
# GOOGLE COLAB MEIN FLASK GUI SETUP
# ============================================

# Ngrok install karo (Colab ke liye zaroori hai)
!pip install flask flask-cors pyngrok numpy pandas librosa tensorflow joblib matplotlib seaborn
!pip install ngrok

# Ngrok account setup (free)
from pyngrok import ngrok

# Ngrok auth token - FREE ACCOUNT BANAO AUR TOKEN LO
# Website: https://dashboard.ngrok.com/auth
NGROK_AUTH_TOKEN = "AUTH_TOKEN"  # Apna token yahan paste karo

# Token set karo
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print(" Ngrok configured successfully!")

 Ngrok configured successfully!


In [80]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import joblib
import tensorflow as tf
from flask import Flask, render_template_string, request, jsonify
from flask_cors import CORS
import threading
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io
import base64
warnings.filterwarnings('ignore')

# ============================================
# FLASK APP INITIALIZATION
# ============================================

app = Flask(__name__)
CORS(app)

# Parameters
SAMPLE_RATE = 16000
DURATION = 3
FEATURE_SIZE = 29

# Global variables
model = None
scaler = None

# Load your trained model
try:
    if os.path.exists('speech_model.pkl'):
        model = joblib.load('speech_model.pkl')
        print("✅ Model loaded successfully!")
    else:
        print("⚠️ Model file not found. Please train model first.")
except Exception as e:
    print(f"❌ Error loading model: {e}")

try:
    if os.path.exists('scaler.pkl'):
        scaler = joblib.load('scaler.pkl')
        print("✅ Scaler loaded successfully!")
except Exception as e:
    print(f"❌ Error loading scaler: {e}")

# ============================================
# FEATURE EXTRACTION
# ============================================

def get_features(file_path, duration=3, sr=16000):
    """Extract features from audio"""
    try:
        y, _ = librosa.load(file_path, sr=sr, duration=duration)

        if len(y) < duration * sr:
            y = np.pad(y, (0, duration * sr - len(y)))
        else:
            y = y[:duration * sr]

        y, _ = librosa.effects.trim(y, top_db=20)

        if len(y) == 0:
            return np.zeros(FEATURE_SIZE)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        features = []
        for i in range(13):
            features.append(np.mean(mfcc[i]))
            features.append(np.std(mfcc[i]))

        pitches, _ = librosa.piptrack(y=y, sr=sr)
        pitch_vals = pitches[pitches > 0]
        if len(pitch_vals) > 0:
            features.append(np.mean(pitch_vals))
            features.append(np.std(pitch_vals))
        else:
            features.extend([0, 0])

        energy = np.sum(y**2) / len(y)
        features.append(energy)

        return np.array(features)

    except Exception as e:
        print(f"Error: {e}")
        return np.zeros(FEATURE_SIZE)

def generate_audio_plots(file_path):
    """Generate analysis plots for uploaded audio file"""
    try:
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)

        if len(y) < SAMPLE_RATE * DURATION:
            y = np.pad(y, (0, SAMPLE_RATE * DURATION - len(y)))
        else:
            y = y[:SAMPLE_RATE * DURATION]

        plots = {}

        # 1. Waveform Plot
        fig1, ax1 = plt.subplots(figsize=(12, 4))
        time = np.arange(len(y)) / sr
        ax1.plot(time, y, color='#2dd4bf', linewidth=1.5)
        ax1.set_xlabel('Time (seconds)', fontsize=12)
        ax1.set_ylabel('Amplitude', fontsize=12)
        ax1.set_title('Audio Waveform - Time Domain Analysis', fontsize=14, fontweight='bold')
        ax1.set_facecolor('#f8fafc')
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim(0, DURATION)

        buf1 = io.BytesIO()
        plt.tight_layout()
        plt.savefig(buf1, format='png', dpi=100, bbox_inches='tight', facecolor='white')
        buf1.seek(0)
        plots['waveform'] = base64.b64encode(buf1.getvalue()).decode('utf-8')
        plt.close(fig1)

        # 2. Spectrogram Plot
        fig2, ax2 = plt.subplots(figsize=(12, 5))
        D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
        img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=ax2, cmap='viridis')
        ax2.set_title('Spectrogram - Frequency vs Time', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Time (seconds)', fontsize=12)
        ax2.set_ylabel('Frequency (Hz)', fontsize=12)
        plt.colorbar(img, ax=ax2, format='%+2.0f dB')

        buf2 = io.BytesIO()
        plt.tight_layout()
        plt.savefig(buf2, format='png', dpi=100, bbox_inches='tight', facecolor='white')
        buf2.seek(0)
        plots['spectrogram'] = base64.b64encode(buf2.getvalue()).decode('utf-8')
        plt.close(fig2)

        # 3. MFCC Plot
        fig3, ax3 = plt.subplots(figsize=(12, 5))
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        img = librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=ax3, cmap='coolwarm')
        ax3.set_title('MFCC Features - Mel-frequency Cepstral Coefficients', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Time (seconds)', fontsize=12)
        ax3.set_ylabel('MFCC Coefficients', fontsize=12)
        plt.colorbar(img, ax=ax3)

        buf3 = io.BytesIO()
        plt.tight_layout()
        plt.savefig(buf3, format='png', dpi=100, bbox_inches='tight', facecolor='white')
        buf3.seek(0)
        plots['mfcc'] = base64.b64encode(buf3.getvalue()).decode('utf-8')
        plt.close(fig3)

        # 4. Frequency Spectrum Plot
        fig4, ax4 = plt.subplots(figsize=(12, 4))
        fft_vals = np.abs(np.fft.fft(y))
        fft_freqs = np.fft.fftfreq(len(y), 1/sr)
        ax4.plot(fft_freqs[:len(fft_freqs)//2], fft_vals[:len(fft_vals)//2], color='#764ba2', linewidth=1.5)
        ax4.set_xlabel('Frequency (Hz)', fontsize=12)
        ax4.set_ylabel('Magnitude', fontsize=12)
        ax4.set_title('Frequency Spectrum - FFT Analysis', fontsize=14, fontweight='bold')
        ax4.set_xlim(0, 8000)
        ax4.grid(True, alpha=0.3)
        ax4.fill_between(fft_freqs[:len(fft_freqs)//2], 0, fft_vals[:len(fft_vals)//2], alpha=0.3, color='#764ba2')

        buf4 = io.BytesIO()
        plt.tight_layout()
        plt.savefig(buf4, format='png', dpi=100, bbox_inches='tight', facecolor='white')
        buf4.seek(0)
        plots['spectrum'] = base64.b64encode(buf4.getvalue()).decode('utf-8')
        plt.close(fig4)

        # 5. Pitch Contour Plot
        fig5, ax5 = plt.subplots(figsize=(12, 4))
        pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
        pitch_values = []
        time_values = []
        hop_length = 512
        for i in range(pitches.shape[1]):
            index = magnitudes[:, i].argmax()
            pitch = pitches[index, i]
            if pitch > 0 and pitch < 500:
                pitch_values.append(pitch)
                time_values.append(i * hop_length / sr)

        if len(pitch_values) > 0:
            ax5.plot(time_values, pitch_values, color='#10b981', linewidth=2, marker='o', markersize=4, alpha=0.7)
            ax5.set_ylim(50, 400)
        else:
            ax5.text(0.5, 0.5, 'No clear pitch detected', transform=ax5.transAxes,
                    ha='center', va='center', fontsize=12, color='gray')

        ax5.set_xlabel('Time (seconds)', fontsize=12)
        ax5.set_ylabel('Frequency (Hz)', fontsize=12)
        ax5.set_title('Pitch Contour - Fundamental Frequency (F0)', fontsize=14, fontweight='bold')
        ax5.grid(True, alpha=0.3)

        buf5 = io.BytesIO()
        plt.tight_layout()
        plt.savefig(buf5, format='png', dpi=100, bbox_inches='tight', facecolor='white')
        buf5.seek(0)
        plots['pitch'] = base64.b64encode(buf5.getvalue()).decode('utf-8')
        plt.close(fig5)

        return plots

    except Exception as e:
        print(f"Plot generation error: {e}")
        return None

# ============================================
# PREDICTION FUNCTION
# ============================================

def predict_audio(audio_path):
    """Predict audio file"""
    try:
        features = get_features(audio_path)

        if np.sum(np.abs(features)) == 0:
            return "Error", 0

        features_scaled = scaler.transform(features.reshape(1, -1))
        probability = float(model.predict(features_scaled, verbose=0)[0][0])

        if probability > 0.5:
            prediction = "Dysarthric"
            confidence = probability * 100
        else:
            prediction = "Control"
            confidence = (1 - probability) * 100

        return prediction, confidence, probability * 100

    except Exception as e:
        print(f"Prediction error: {e}")
        return "Error", 0, 0

# ============================================
# HTML GUI TEMPLATE WITH DIRECT IMAGE URL
# ============================================

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dysarthric Speech Detection System | Medical AI</title>
    <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;500;600;700&display=swap" rel="stylesheet">
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Poppins', sans-serif;
            min-height: 100vh;
            position: relative;
            /* Direct image URL from Cyient */
            background-image: url('https://www.cyient.com/hubfs/medical%20device/Mask%20group%20-%202025-02-03T171604.177.jpg');
            background-size: cover;
            background-position: center;
            background-repeat: no-repeat;
            background-attachment: fixed;
        }

        /* Gradient Overlay for better text readability */
        body::before {
            content: "";
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: linear-gradient(135deg,
                rgba(12, 74, 110, 0.75) 0%,
                rgba(30, 58, 138, 0.75) 25%,
                rgba(91, 33, 182, 0.75) 50%,
                rgba(126, 34, 206, 0.75) 75%,
                rgba(162, 28, 175, 0.75) 100%);
            pointer-events: none;
            z-index: 0;
        }

        /* Medical Cross Pattern Overlay */
        body::after {
            content: "⚕️⚕️⚕️";
            position: fixed;
            bottom: 10px;
            right: 10px;
            font-size: 30px;
            opacity: 0.3;
            pointer-events: none;
            z-index: 0;
        }

        .container {
            max-width: 1400px;
            margin: 0 auto;
            position: relative;
            z-index: 1;
            padding: 20px;
        }

        /* Header with medical theme */
        .header {
            text-align: center;
            color: white;
            margin-bottom: 30px;
            padding: 25px;
            background: rgba(255, 255, 255, 0.15);
            backdrop-filter: blur(10px);
            border-radius: 20px;
            border: 1px solid rgba(255, 255, 255, 0.2);
            animation: slideDown 0.6s ease;
        }

        @keyframes slideDown {
            from {
                opacity: 0;
                transform: translateY(-30px);
            }
            to {
                opacity: 1;
                transform: translateY(0);
            }
        }

        .header h1 {
            font-size: 2.5rem;
            margin-bottom: 10px;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 15px;
        }

        .header p {
            font-size: 1.1rem;
            opacity: 0.95;
        }

        .medical-badge {
            background: #2dd4bf;
            color: #0f766e;
            padding: 5px 15px;
            border-radius: 20px;
            display: inline-block;
            font-size: 0.9rem;
            margin-top: 10px;
        }

        /* Cards */
        .card {
            background: rgba(255, 255, 255, 0.95);
            backdrop-filter: blur(10px);
            border-radius: 20px;
            padding: 25px;
            margin-bottom: 25px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.15);
            transition: transform 0.3s ease, box-shadow 0.3s ease;
            border: 1px solid rgba(255, 255, 255, 0.2);
            animation: fadeInUp 0.6s ease;
        }

        @keyframes fadeInUp {
            from {
                opacity: 0;
                transform: translateY(30px);
            }
            to {
                opacity: 1;
                transform: translateY(0);
            }
        }

        .card:hover {
            transform: translateY(-5px);
            box-shadow: 0 15px 50px rgba(0,0,0,0.2);
        }

        .card-title {
            font-size: 1.5rem;
            margin-bottom: 20px;
            color: #1e293b;
            border-left: 4px solid #2dd4bf;
            padding-left: 15px;
            display: flex;
            align-items: center;
            gap: 10px;
        }

        /* Upload Area */
        .upload-area {
            border: 2px dashed #2dd4bf;
            border-radius: 15px;
            padding: 40px;
            text-align: center;
            cursor: pointer;
            transition: all 0.3s ease;
            background: rgba(248, 250, 252, 0.9);
        }

        .upload-area:hover {
            background: #e0f2fe;
            border-color: #0ea5e9;
            transform: scale(1.02);
        }

        .upload-area.drag-over {
            background: #bae6fd;
            border-color: #0284c7;
        }

        .upload-icon {
            font-size: 48px;
            color: #2dd4bf;
            margin-bottom: 15px;
        }

        /* File Info */
        .file-info {
            background: #f1f5f9;
            border-radius: 10px;
            padding: 15px;
            margin-top: 15px;
            display: none;
        }

        /* Button */
        .btn-predict {
            background: linear-gradient(135deg, #2dd4bf, #0ea5e9);
            color: white;
            border: none;
            padding: 12px 30px;
            border-radius: 25px;
            font-size: 16px;
            cursor: pointer;
            width: 100%;
            margin-top: 15px;
            transition: all 0.3s ease;
            font-weight: 600;
        }

        .btn-predict:hover:not(:disabled) {
            transform: translateY(-2px);
            box-shadow: 0 5px 20px rgba(45, 212, 191, 0.4);
        }

        .btn-predict:disabled {
            opacity: 0.6;
            cursor: not-allowed;
        }

        /* Result Section */
        .result-section {
            display: none;
            animation: slideIn 0.5s ease;
        }

        @keyframes slideIn {
            from {
                opacity: 0;
                transform: translateY(-20px);
            }
            to {
                opacity: 1;
                transform: translateY(0);
            }
        }

        .result-badge {
            text-align: center;
            padding: 30px;
            border-radius: 15px;
            margin-bottom: 20px;
        }

        .result-dysarthric {
            background: linear-gradient(135deg, #ef4444, #dc2626);
            color: white;
        }

        .result-control {
            background: linear-gradient(135deg, #10b981, #059669);
            color: white;
        }

        .prediction-text {
            font-size: 2rem;
            font-weight: bold;
            margin: 20px 0;
        }

        /* Confidence Meter */
        .confidence-meter {
            background: #f1f5f9;
            border-radius: 15px;
            padding: 20px;
            margin: 20px 0;
        }

        .progress-bar-container {
            background: #e2e8f0;
            border-radius: 10px;
            overflow: hidden;
            height: 40px;
            margin: 10px 0;
        }

        .progress-bar-fill {
            background: linear-gradient(90deg, #2dd4bf, #0ea5e9);
            height: 100%;
            display: flex;
            align-items: center;
            justify-content: center;
            color: white;
            font-weight: bold;
            transition: width 0.5s ease;
        }

        /* Metrics Grid */
        .metrics-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px;
            margin: 20px 0;
        }

        .metric-card {
            background: linear-gradient(135deg, #f8fafc, #f1f5f9);
            padding: 20px;
            border-radius: 15px;
            text-align: center;
            border: 1px solid #e2e8f0;
            transition: all 0.3s ease;
        }

        .metric-card:hover {
            transform: translateY(-3px);
            box-shadow: 0 5px 15px rgba(0,0,0,0.1);
        }

        .metric-value {
            font-size: 2rem;
            font-weight: bold;
            background: linear-gradient(135deg, #2dd4bf, #0ea5e9);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
        }

        .metric-label {
            color: #475569;
            margin-top: 5px;
            font-size: 0.9rem;
        }

        /* Tabs */
        .tabs {
            display: flex;
            gap: 10px;
            margin-bottom: 20px;
            flex-wrap: wrap;
        }

        .tab-btn {
            background: #f1f5f9;
            border: none;
            padding: 10px 20px;
            border-radius: 25px;
            cursor: pointer;
            transition: all 0.3s ease;
            font-weight: 500;
        }

        .tab-btn:hover {
            background: #e2e8f0;
            transform: translateY(-2px);
        }

        .tab-btn.active {
            background: linear-gradient(135deg, #2dd4bf, #0ea5e9);
            color: white;
        }

        .tab-content {
            display: none;
        }

        .tab-content.active {
            display: block;
            animation: fadeIn 0.5s ease;
        }

        @keyframes fadeIn {
            from { opacity: 0; }
            to { opacity: 1; }
        }

        .analysis-card {
            background: #ffffff;
            border-radius: 15px;
            padding: 15px;
            border: 1px solid #e2e8f0;
        }

        .analysis-card h4 {
            color: #1e293b;
            margin-bottom: 15px;
            display: flex;
            align-items: center;
            gap: 8px;
        }

        .analysis-card img {
            width: 100%;
            border-radius: 10px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }

        /* Loading Spinner */
        .loading {
            display: none;
            text-align: center;
            padding: 20px;
        }

        .spinner {
            border: 4px solid #e2e8f0;
            border-top: 4px solid #2dd4bf;
            border-radius: 50%;
            width: 50px;
            height: 50px;
            animation: spin 1s linear infinite;
            margin: 0 auto 15px auto;
        }

        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }

        audio {
            width: 100%;
            margin-top: 15px;
            border-radius: 10px;
        }

        /* Footer */
        .footer {
            text-align: center;
            color: white;
            padding: 20px;
            font-size: 0.85rem;
            opacity: 0.8;
        }

        /* Responsive */
        @media (max-width: 768px) {
            .container {
                padding: 10px;
            }
            .header h1 {
                font-size: 1.5rem;
            }
            .prediction-text {
                font-size: 1.2rem;
            }
            .metrics-grid {
                grid-template-columns: 1fr;
            }
            .tabs {
                justify-content: center;
            }
            .tab-btn {
                padding: 6px 12px;
                font-size: 12px;
            }
        }
    </style>
</head>
<body>
    <div class="container">
        <!-- Header -->
        <div class="header">
            <h1>
                <span>🏥</span>
                Dysarthric Speech Detection System
                <span>⚕️</span>
            </h1>
            <p>AI-Powered Medical Speech Analysis Tool for Dysarthria Detection</p>
            <div class="medical-badge">
                <span>🩺</span> Medical Diagnostic Assistant <span>📊</span>
            </div>
        </div>

        <!-- Upload Card -->
        <div class="card">
            <div class="card-title">
                <span>📁</span> Upload Audio File for Analysis
            </div>

            <div class="upload-area" id="uploadArea">
                <div class="upload-icon">🎵</div>
                <h3>Click or Drag & Drop Audio File</h3>
                <p>Supported formats: WAV, MP3, M4A (Max 16MB)</p>
                <input type="file" id="fileInput" accept=".wav,.mp3,.m4a" style="display: none;">
            </div>

            <div class="file-info" id="fileInfo">
                <strong id="fileName"></strong><br>
                <small id="fileSize"></small>
                <audio id="audioPlayer" controls></audio>
            </div>

            <button class="btn-predict" id="predictBtn" disabled>
                <span>🧠</span> Analyze Speech for Dysarthria
            </button>

            <div class="loading" id="loading">
                <div class="spinner"></div>
                <p>Analyzing audio features...</p>
                <p style="font-size: 12px;">Extracting MFCC, Pitch, Frequency features and generating graphs...</p>
            </div>
        </div>

        <!-- Result Section -->
        <div class="result-section" id="resultSection">
            <!-- Prediction Result Card -->
            <div class="card">
                <div class="result-badge" id="resultBadge">
                    <div class="prediction-text" id="predictionText"></div>
                    <div id="messageText"></div>
                </div>

                <div class="confidence-meter">
                    <h3><span>📊</span> Confidence Score</h3>
                    <div class="progress-bar-container">
                        <div class="progress-bar-fill" id="confidenceBar" style="width: 0%;">
                            <span id="confidenceValue">0%</span>
                        </div>
                    </div>
                </div>

                <div class="metrics-grid">
                    <div class="metric-card">
                        <div class="metric-value" id="accuracy">93%</div>
                        <div class="metric-label">Model Accuracy</div>
                        <small style="color: #10b981;">✓ High Performance</small>
                    </div>
                    <div class="metric-card">
                        <div class="metric-value" id="f1score">92%</div>
                        <div class="metric-label">F1 Score</div>
                        <small style="color: #10b981;">✓ Balanced</small>
                    </div>
                    <div class="metric-card">
                        <div class="metric-value" id="auc">93%</div>
                        <div class="metric-label">AUC Score</div>
                        <small style="color: #10b981;">✓ Excellent</small>
                    </div>
                </div>
            </div>

            <!-- Audio Analysis Graphs -->
            <div class="card">
                <div class="card-title">
                    <span>📈</span> Audio Signal Analysis - Uploaded File
                </div>

                <div class="tabs" id="analysisTabs">
                    <button class="tab-btn active" data-tab="waveform">📊 Waveform</button>
                    <button class="tab-btn" data-tab="spectrogram">🎨 Spectrogram</button>
                    <button class="tab-btn" data-tab="mfcc">🎵 MFCC Features</button>
                    <button class="tab-btn" data-tab="spectrum">📡 Frequency Spectrum</button>
                    <button class="tab-btn" data-tab="pitch">🎤 Pitch Contour</button>
                </div>

                <div id="waveform-tab" class="tab-content active">
                    <div class="analysis-card">
                        <h4><span>📊</span> Audio Waveform - Time Domain</h4>
                        <img id="waveformPlot" alt="Waveform Plot" style="width:100%">
                    </div>
                </div>

                <div id="spectrogram-tab" class="tab-content">
                    <div class="analysis-card">
                        <h4><span>🎨</span> Spectrogram - Frequency Spectrum over Time</h4>
                        <img id="spectrogramPlot" alt="Spectrogram Plot" style="width:100%">
                    </div>
                </div>

                <div id="mfcc-tab" class="tab-content">
                    <div class="analysis-card">
                        <h4><span>🎵</span> MFCC Features - Mel-frequency Cepstral Coefficients</h4>
                        <img id="mfccPlot" alt="MFCC Plot" style="width:100%">
                    </div>
                </div>

                <div id="spectrum-tab" class="tab-content">
                    <div class="analysis-card">
                        <h4><span>📡</span> Frequency Spectrum - FFT Analysis</h4>
                        <img id="spectrumPlot" alt="Spectrum Plot" style="width:100%">
                    </div>
                </div>

                <div id="pitch-tab" class="tab-content">
                    <div class="analysis-card">
                        <h4><span>🎤</span> Pitch Contour - Fundamental Frequency (F0)</h4>
                        <img id="pitchPlot" alt="Pitch Plot" style="width:100%">
                    </div>
                </div>
            </div>
        </div>

        <div class="footer">
            <p>Powered by Deep Learning | Medical AI Assistant | Real-time Speech Analysis</p>
        </div>
    </div>

    <script>
        const uploadArea = document.getElementById('uploadArea');
        const fileInput = document.getElementById('fileInput');
        const predictBtn = document.getElementById('predictBtn');
        const fileInfo = document.getElementById('fileInfo');
        const fileName = document.getElementById('fileName');
        const fileSize = document.getElementById('fileSize');
        const audioPlayer = document.getElementById('audioPlayer');
        const loading = document.getElementById('loading');
        const resultSection = document.getElementById('resultSection');
        const resultBadge = document.getElementById('resultBadge');
        const predictionText = document.getElementById('predictionText');
        const messageText = document.getElementById('messageText');
        const confidenceBar = document.getElementById('confidenceBar');
        const confidenceValue = document.getElementById('confidenceValue');

        let selectedFile = null;

        // Tab switching
        const tabs = document.querySelectorAll('.tab-btn');
        tabs.forEach(tab => {
            tab.addEventListener('click', () => {
                const tabId = tab.getAttribute('data-tab');
                tabs.forEach(t => t.classList.remove('active'));
                tab.classList.add('active');

                document.querySelectorAll('.tab-content').forEach(content => {
                    content.classList.remove('active');
                });
                document.getElementById(`${tabId}-tab`).classList.add('active');
            });
        });

        uploadArea.addEventListener('click', () => fileInput.click());

        uploadArea.addEventListener('dragover', (e) => {
            e.preventDefault();
            uploadArea.classList.add('drag-over');
        });

        uploadArea.addEventListener('dragleave', () => {
            uploadArea.classList.remove('drag-over');
        });

        uploadArea.addEventListener('drop', (e) => {
            e.preventDefault();
            uploadArea.classList.remove('drag-over');
            const files = e.dataTransfer.files;
            if (files.length > 0) handleFile(files[0]);
        });

        fileInput.addEventListener('change', (e) => {
            if (e.target.files.length > 0) handleFile(e.target.files[0]);
        });

        function handleFile(file) {
            const validExtensions = ['wav', 'mp3', 'm4a'];
            const extension = file.name.split('.').pop().toLowerCase();

            if (!validExtensions.includes(extension)) {
                alert('Please upload WAV, MP3, or M4A file');
                return;
            }

            if (file.size > 16 * 1024 * 1024) {
                alert('File size must be less than 16MB');
                return;
            }

            selectedFile = file;
            fileName.textContent = file.name;
            fileSize.textContent = `${(file.size / 1024).toFixed(2)} KB`;

            const audioUrl = URL.createObjectURL(file);
            audioPlayer.src = audioUrl;

            fileInfo.style.display = 'block';
            predictBtn.disabled = false;
            resultSection.style.display = 'none';
        }

        predictBtn.addEventListener('click', async () => {
            if (!selectedFile) return;

            const formData = new FormData();
            formData.append('audio', selectedFile);

            predictBtn.disabled = true;
            loading.style.display = 'block';
            resultSection.style.display = 'none';

            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    body: formData
                });

                const data = await response.json();

                if (response.ok && data.success) {
                    const isDysarthric = data.prediction === 'Dysarthric';

                    resultBadge.className = `result-badge ${isDysarthric ? 'result-dysarthric' : 'result-control'}`;
                    predictionText.innerHTML = isDysarthric ? '⚠️ Dysarthric Speech Detected' : '✅ Normal Speech Detected';

                    const confidence = data.confidence;
                    confidenceBar.style.width = `${confidence}%`;
                    confidenceValue.textContent = `${confidence}%`;

                    let message = '';
                    if (isDysarthric) {
                        if (confidence > 80) message = '🏥 High confidence: Speech shows strong characteristics of dysarthria. Please consult a healthcare professional.';
                        else if (confidence > 60) message = '🩺 Moderate confidence: Speech shows some characteristics of dysarthria. Consider professional evaluation.';
                        else message = '📋 Low confidence: Speech shows mild characteristics of dysarthria. Further testing recommended.';
                    } else {
                        if (confidence > 80) message = '✅ High confidence: Speech patterns appear normal. No significant dysarthric indicators detected.';
                        else if (confidence > 60) message = '✅ Moderate confidence: Speech patterns appear mostly normal. Minimal indicators detected.';
                        else message = '📝 Low confidence: Speech appears normal, but consider retesting with clearer audio.';
                    }
                    messageText.textContent = message;

                    if (data.plots) {
                        if (data.plots.waveform) document.getElementById('waveformPlot').src = `data:image/png;base64,${data.plots.waveform}`;
                        if (data.plots.spectrogram) document.getElementById('spectrogramPlot').src = `data:image/png;base64,${data.plots.spectrogram}`;
                        if (data.plots.mfcc) document.getElementById('mfccPlot').src = `data:image/png;base64,${data.plots.mfcc}`;
                        if (data.plots.spectrum) document.getElementById('spectrumPlot').src = `data:image/png;base64,${data.plots.spectrum}`;
                        if (data.plots.pitch) document.getElementById('pitchPlot').src = `data:image/png;base64,${data.plots.pitch}`;
                    }

                    resultSection.style.display = 'block';
                } else {
                    alert('Error: ' + (data.error || 'Prediction failed'));
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to analyze audio. Please try again.');
            } finally {
                loading.style.display = 'none';
                predictBtn.disabled = false;
            }
        });

        document.getElementById('accuracy').textContent = '93%';
        document.getElementById('f1score').textContent = '92%';
        document.getElementById('auc').textContent = '93%';
    </script>
</body>
</html>
"""

# ============================================
# FLASK ROUTES
# ============================================

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        if 'audio' not in request.files:
            return jsonify({'error': 'No audio file'}), 400

        file = request.files['audio']

        if file.filename == '':
            return jsonify({'error': 'No file selected'}), 400

        temp_path = '/tmp/temp_audio.wav'
        file.save(temp_path)

        plots = generate_audio_plots(temp_path)
        prediction, confidence, probability = predict_audio(temp_path)

        os.remove(temp_path)

        if prediction == "Error":
            return jsonify({'error': 'Could not process audio'}), 400

        return jsonify({
            'success': True,
            'prediction': prediction,
            'confidence': round(confidence, 2),
            'probability': round(probability, 2),
            'plots': plots
        })

    except Exception as e:
        print(f"Error: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/model_info', methods=['GET'])
def model_info():
    return jsonify({
        'success': True,
        'accuracy': 0.93,
        'f1_score': 0.92,
        'auc_score': 0.93
    })

# ============================================
# RUN FLASK WITH NGROK
# ============================================

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()

from pyngrok import ngrok

# Apna ngrok token yahan paste karein
NGROK_AUTH_TOKEN = "AUTH_TOKEN"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(5000)
print(f"\n{'='*60}")
print(f"🏥 DYSARTHRIC SPEECH DETECTION GUI IS LIVE!")
print(f"{'='*60}")
print(f"📱 Open this URL in your browser:")
print(f"🔗 {public_url}")
print(f"{'='*60}")
print(f"\n💡 Features Included:")
print(f"   ✅ Medical device background image from Cyient")
print(f"   ✅ Professional medical theme with overlay")
print(f"   ✅ 5 Audio Analysis Graphs (Waveform, Spectrogram, MFCC, Spectrum, Pitch)")
print(f"   ✅ Real-time prediction with confidence score")
print(f"   ✅ Metrics: 93% Accuracy, 92% F1, 93% AUC")
print(f"\n⚠️ Note: Press Ctrl+C to stop the server")
print(f"{'='*60}\n")

✅ Model loaded successfully!
✅ Scaler loaded successfully!
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: Your account may not run more than 5 endpoints over a single ngrok agent session.\nThe endpoints already running on this session are:\ntn_3DUVHFbNRim4kkefHJxoXe81v14, tn_3DUZ5TPIQ41MHXAXEb9dutLRhtQ, tn_3DUcLdS2h567nCVh3bR2ZU5JA7b, tn_3DUl8tX9gwxHaQWXb9bTUP3pFUX, tn_3DUDnbSctHfkBcTan086E8euDr5.\nUpgrade to a Pay-as-you-go plan at: https://dashboard.ngrok.com/billing/choose-a-plan?plan=paygo\r\n\r\nERR_NGROK_324\r\n"}}
